# LlamaIndex와 AgentCore Memory - 의료 지식 Assistant(단기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 의료 지식 Assistant를 만드는 방법을 살펴봅니다. 하나의 환자 상담 세션 안에서 **단기 메모리**를 유지하여 의료 상담 전반에 걸쳐 환자 증상, 병력, 약물 상호작용, 진단 추론을 기억하도록 하는 데 중점을 둡니다.

## 아키텍처 개요

![LlamaIndex AgentCore Short-Term Memory Architecture](LlamaIndex-AgentCore-STM-Arch.png)

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화 메모리                                                |
| Agent 사용 사례       | 의료 지식 Assistant                                                      |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM 모델           | Anthropic Claude 3.7 Sonnet                                                       |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, LlamaIndex Agent, 의료 분석 도구           |
| 예제 난이도  | 중급                                                                     |

다음 내용을 학습합니다.
- 의료 상담 데이터를 위한 AgentCore Memory 생성
- 의료 workflow에 LlamaIndex 기본 메모리 통합 사용
- 환자 분석을 위한 의료 전용 도구 구축
- 단일 상담 세션 내에서 의료 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 배경

이 예제에서는 의료 제공자가 단일 상담 세션 안에서 환자 사례를 분석하고 약물 상호작용을 확인하며 임상 가이드라인을 검색하도록 돕는 "의료 지식 Assistant"를 만듭니다. Assistant는 AgentCore Memory를 사용하여 상담 전반에 걸쳐 환자 증상, 병력, 약물, 진단 추론에 관한 컨텍스트를 유지합니다.

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델에 대한 액세스

## 1단계: 종속성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 구성 요소 가져오기
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore Memory 구성

의료 Assistant에서 사용할 AgentCore Memory 리소스를 생성하거나 가져옵니다.

In [ ]:
# AgentCore Memory 리소스 생성
region = os.getenv("AWS_REGION", "us-east-1")
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f"MedicalAssistantShortTerm_{int(datetime.now().timestamp())}",
        description="Medical knowledge assistant short-term memory for single consultation context",
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10,
    )
    memory_id = response["id"]
    print(f"✅ Created AgentCore Memory: {memory_id}")
except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 Memory ID로 교체

## 3단계: 의료 분석 도구 구현

의료 상담 작업을 위한 전문 도구를 정의합니다.

In [ ]:
def record_patient_symptoms(symptoms: str, severity: str, duration: str) -> str:
    """Record patient symptoms with severity and duration"""
    print(f"🩺 Recorded symptoms: {symptoms} ({severity} severity, {duration} duration)")
    return f"Recorded patient symptoms: {symptoms}"


def check_drug_interaction(medication1: str, medication2: str, interaction_level: str) -> str:
    """Check drug interaction between medications"""
    print(f"💊 Drug interaction check: {medication1} + {medication2} ({interaction_level} risk)")
    return f"Drug interaction assessed: {medication1} and {medication2}"


def save_vital_signs(temperature: str, blood_pressure: str, heart_rate: str, notes: str) -> str:
    """Save patient vital signs with notes"""
    print(f"📊 Vital signs: Temp {temperature}, BP {blood_pressure}, HR {heart_rate}")
    return "Saved vital signs for patient"


def retrieve_clinical_guideline(condition: str, guideline_type: str, evidence_level: str) -> str:
    """Retrieve clinical guideline for medical condition"""
    print(f"📋 Retrieved {guideline_type} guideline for {condition} (Evidence: {evidence_level})")
    return f"Retrieved clinical guideline for {condition}"


def document_differential_diagnosis(primary_diagnosis: str, alternatives: str, confidence: str) -> str:
    """Document differential diagnosis with confidence level"""
    print(f"🔍 Differential diagnosis: {primary_diagnosis} ({confidence} confidence)")
    return f"Documented differential diagnosis: {primary_diagnosis}"


# Agent용 도구 객체 생성
medical_tools = [
    FunctionTool.from_defaults(fn=record_patient_symptoms),
    FunctionTool.from_defaults(fn=check_drug_interaction),
    FunctionTool.from_defaults(fn=save_vital_signs),
    FunctionTool.from_defaults(fn=retrieve_clinical_guideline),
    FunctionTool.from_defaults(fn=document_differential_diagnosis),
]

## 4단계: LlamaIndex Agent 구현

단기 메모리 컨텍스트를 사용하는 의료 Assistant Agent를 생성합니다.

In [ ]:
# 단기 메모리 구성(단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="medical-provider",
    memory_id=memory_id,
    session_id="consultation-session-today",  # 전체 과정에서 동일한 세션 사용
    namespace="/medical-consultation/",
)

# AgentCore Memory 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 의료 Assistant Agent 생성
medical_agent = FunctionAgent(tools=medical_tools, llm=llm, verbose=True)

print("✅ Medical Knowledge Assistant with short-term memory is ready!")

## 5단계: 단기 메모리 기능 테스트

종합 환자 상담 세션을 통해 의료 Assistant의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 환자 접수 및 초기 평가

In [ ]:
# 환자 세부 정보로 상담 세션 초기화
response = await medical_agent.run(
    "I'm Dr. Emily Chen conducting a consultation for patient John Smith, 45-year-old male. "
    "Record symptoms: 'chest pain, shortness of breath, fatigue' with 'severe' severity and '3 days' duration. "
    "Patient has history of hypertension and diabetes.",
    memory=agentcore_memory,
)

print("🎯 Patient Intake:")
print(response)

### 테스트 2: 활력 징후 기록

In [ ]:
# 임상 컨텍스트와 활력 징후 기록
response = await medical_agent.run(
    "Save vital signs: temperature '99.2°F', blood pressure '165/95 mmHg', heart rate '110 bpm' "
    "with notes 'elevated BP and tachycardia, patient appears diaphoretic and anxious'.",
    memory=agentcore_memory,
)

print("📊 Vital Signs Documentation:")
print(response)

### 테스트 3: 약물 상호작용 분석

In [ ]:
# 현재 복용 약물의 상호작용 확인
response = await medical_agent.run(
    "Check drug interaction between 'Lisinopril 10mg' and 'Metformin 500mg' with 'low' interaction level. "
    "Patient is currently taking both for hypertension and diabetes management.",
    memory=agentcore_memory,
)

print("💊 Drug Interaction Check:")
print(response)

# 새 약물의 잠재적 상호작용 확인
response = await medical_agent.run(
    "Check drug interaction between 'Lisinopril 10mg' and 'Nitroglycerin sublingual' with 'moderate' interaction level. "
    "Considering nitroglycerin for chest pain management.",
    memory=agentcore_memory,
)

print("💊 Additional Drug Check:")
print(response)

### 테스트 4: 환자 컨텍스트 회상

In [ ]:
# 환자 정보 및 활력 징후 회상 테스트
response = await medical_agent.run(
    "What patient am I consulting with? What are their presenting symptoms, vital signs, and current medications?",
    memory=agentcore_memory,
)

print("🧠 Patient Context Recall:")
print(response)
print("\n✅ Expected: John Smith, 45M, chest pain/SOB/fatigue, elevated BP 165/95, Lisinopril/Metformin")

### 테스트 5: 임상 가이드라인 검색

In [ ]:
# 증상에 따른 임상 가이드라인 검색
response = await medical_agent.run(
    "Retrieve clinical guideline for 'acute chest pain' with 'diagnostic protocol' type and 'Level A' evidence level. "
    "Need to evaluate this patient's chest pain systematically.",
    memory=agentcore_memory,
)

print("📋 Clinical Guideline Retrieval:")
print(response)

### 테스트 6: 감별 진단 기록

In [ ]:
# 추론과 함께 감별 진단 기록
response = await medical_agent.run(
    "Document differential diagnosis: primary 'Acute Coronary Syndrome' with alternatives "
    "'pulmonary embolism, aortic dissection, anxiety disorder' and 'high' confidence level. "
    "Based on chest pain, elevated vitals, and cardiac risk factors.",
    memory=agentcore_memory,
)

print("🔍 Differential Diagnosis:")
print(response)

### 테스트 7: 종합 임상 추론

In [ ]:
# 종합 임상 추론 테스트
response = await medical_agent.run(
    "Based on John's symptoms, vital signs, and medical history, why did I consider Acute Coronary Syndrome? "
    "What specific clinical indicators support this diagnosis?",
    memory=agentcore_memory,
)

print("🤔 Clinical Reasoning Test:")
print(response)
print("\n✅ Expected: Chest pain + SOB + elevated BP/HR + diabetes/HTN history = ACS risk factors")

### 테스트 8: 약물 상호작용 회상

In [ ]:
# 약물 상호작용 기억 테스트
response = await medical_agent.run(
    "What drug interactions have I checked for this patient? Which combination had moderate risk and why?",
    memory=agentcore_memory,
)

print("💊 Drug Interaction Recall:")
print(response)
print("\n✅ Expected: Lisinopril+Metformin (low risk), Lisinopril+Nitroglycerin (moderate risk)")

### 테스트 9: 치료 계획 종합

In [ ]:
# 통합 치료 계획 테스트
response = await medical_agent.run(
    "Based on my differential diagnosis and drug interaction checks, what treatment considerations "
    "should I keep in mind for John? Include medication interactions and clinical guidelines.",
    memory=agentcore_memory,
)

print("🏥 Treatment Planning:")
print(response)
print("\n✅ Expected: ACS protocol, monitor Lisinopril+Nitroglycerin interaction, consider cardiac workup")

In [ ]:
# 종합 사례 요약
response = await medical_agent.run(
    "Provide a complete case summary: patient demographics, presenting symptoms, vital signs, "
    "current medications, drug interactions checked, differential diagnosis, and clinical guidelines retrieved.",
    memory=agentcore_memory,
)

print("📋 Complete Case Summary:")
print(response)
print("\n✅ Expected: Full consultation details with all recorded information")

## 6단계: 세션 경계 테스트

별도의 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다.

In [ ]:
# 별도의 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="medical-provider",
    memory_id=memory_id,
    session_id="different-consultation-session",  # 서로 다른 세션 ID
    namespace="/medical-consultation/",
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
response = await medical_agent.run(
    "What patients am I consulting with today? What symptoms and vital signs have I recorded?",
    memory=new_session_memory,
)

print("🚧 Session Boundary Test (Different Session):")
print(response)
print("\n✅ Expected: Limited or no recall from previous session (short-term memory boundary)")

In [ ]:
# 지속성을 검증하기 위해 원래 세션으로 복귀
response = await medical_agent.run(
    "Back in my original consultation - what were John Smith's exact vital signs and primary diagnosis?",
    memory=agentcore_memory,  # 원래 세션 메모리
)

print("🔄 Original Session Return:")
print(response)
print("\n✅ Expected: Full recall of BP 165/95, HR 110, ACS diagnosis")

## 🧪 자동 테스트 검증
다음 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# 검증 함수를 인라인으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # 실질적인 응답인지 확인("I don't know"만 있는 응답 제외)
        has_content = len(response) > 50
        # 메모리 관련 표현 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - Agent가 논의한 내용을 기억하는가?
response1 = await medical_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: 세션 메모리 - Agent가 컨텍스트를 유지하는가?
response2 = await medical_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - Agent가 이전 컨텍스트와 연결할 수 있는가?
response3 = await medical_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

### 테스트 10: 종합 사례 요약

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

✅ **단기 메모리 통합**: LlamaIndex에서 AgentCore Memory를 사용하여 세션 범위의 의료 상담 제공

✅ **의료 전용 도구**: 환자 증상 추적, 약물 상호작용 확인, 임상 가이드라인 검색

✅ **임상 추론**: Assistant가 환자 세부 정보, 활력 징후, 진단 추론을 기억

✅ **약물 안전 관리**: 종합적인 약물 상호작용 추적 및 평가

✅ **세션 경계**: 서로 다른 환자 상담 세션 간의 메모리 격리

✅ **근거 중심 의학**: 임상 가이드라인 통합 및 감별 진단 기록

의료 지식 Assistant는 단기 메모리를 통해 하나의 상담 세션 안에서 종합적인 환자 진료를 지원하는 동시에 서로 다른 환자 진료 사이의 경계를 명확히 유지하는 방법을 보여 줍니다.

## 리소스 정리

이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# AgentCore Memory 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")